In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Jupyter may start the kernel in the repository root or in notebooks/.
PROJECT_ROOT = Path.cwd() if Path("data").is_dir() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import random
import math
import seaborn as sns

In [ ]:
data=pd.read_csv(DATA_DIR / "train.csv",index_col="Id")
prev_data=pd.read_csv(DATA_DIR / "prev.csv",index_col="Id")

data.head()

In [ ]:
prev_data.head()

# Algorithme de k-plus proches voisins

In [ ]:
df=data.copy()
df_p=prev_data.copy()

In [ ]:
# On enlève les colonnes non numériques → on garde seulement les variables utiles pour calculer les distances
df_num = df.drop(columns=['espece', 'ile_origine', 'sexe'])
dfp_num = df_p.drop(columns=['ile_origine', 'sexe', 'Usage'])

In [ ]:
dfp_num = dfp_num[df_num.columns] #On aligne les colonnes du test avec celles du train sinon, mat_desc-x ne va pas marcher

In [ ]:
mean_train = df_num.mean() # Moyenne de chaque colonne (sur le train)
std_train = df_num.std() # Écart-type de chaque colonne (sur le train)

normalize_df_num = (df_num - mean_train) / std_train # On normalise les données d'entraînement (centrer + réduire)
normalize_dfp_num = (dfp_num - mean_train) / std_train # On normalise les données de test avec les mêmes valeurs!

In [ ]:
base_class=df['espece'] # Ce sont les labels (les classes à prédire)

In [ ]:
x_train=normalize_df_num.to_numpy()
y_train=base_class.to_numpy()

x_test=normalize_dfp_num.to_numpy()
#y_test est ce qu'on doit prédire, il exste pas dans prev

In [ ]:
#on sépqre les données de data en test and train; pour qu'on puisse tester notre code
X_test_sep=x_train[:500]
Y_test_sep=y_train[:500]
X_train_sep=x_train[500:]
Y_train_sep=y_train[500:]


In [ ]:
def classe_majoritaire(vec):
    vals, counts = np.unique(vec, return_counts=True)  # vals = classes uniques, counts = combien de fois chaque classe apparaît
    return vals[np.argmax(counts)]  # On retourne la classe la plus fréquente

In [ ]:
def classifie_kppv(k, mat_desc, mat_class, x):
    distances = np.sum((mat_desc - x)**2, axis=1) 
    # 1. mat_desc - x → différence entre x et chaque point du train
    # 2. **2 → on met au carré chaque différence (élément par élément, c'est pas un produit matriciel)
    # 3. sum(axis=1) → on additionne chaque ligne → distance totale pour chaque point

    idx_plus_proches = np.argsort(distances)[:k] # On trie les distances et on prend les indices des k plus petites (les plus proches)

    classes_voisines = mat_class[idx_plus_proches] # On récupère les classes des k voisins les plus proches
    
    return classe_majoritaire(classes_voisines) # On retourne la classe la plus fréquente parmi ces voisins


In [ ]:
classifie_kppv(5, X_train_sep,Y_train_sep , X_test_sep[0])

In [ ]:
Y_test_sep_pred = np.array([classifie_kppv(17, X_train_sep, Y_train_sep, X_test_sep[i]) for i in range(X_test_sep.shape[0])])

In [ ]:
(Y_test_sep_pred==Y_test_sep).mean()*100 #on test l'accuracy

In [ ]:
k_max = 20 # On fixe la valeur maximale de k (nombre de voisins à tester)
accuracy = [] # Liste vide pour stocker les scores de précision pour chaque k

for k in range(1, k_max):
    # On teste toutes les valeurs de k de 1 à 19
    
    y_pred = np.array([classifie_kppv(k, X_train_sep, Y_train_sep, X_test_sep[j])for j in range(X_test_sep.shape[0])])
    # Pour chaque point du test :
    # - on applique k-NN
    # - on récupère la classe prédite
    # → y_pred contient toutes les prédictions
    
    acc = (y_pred == Y_test_sep).mean()*100 # On compare les prédictions avec les vraies classes et .mean() donne le pourcentage de bonnes réponses
    accuracy.append(acc) # On ajoute ce score dans la liste

plt.plot(range(1, k_max), accuracy, color="red")
plt.xlabel("k")
plt.ylabel("Accuracy (%)")
plt.show()

In [ ]:
k_max = 20
taux_erreur = []

for k in range(1, k_max):
    
    y_pred = np.array([classifie_kppv(k, X_train_sep, Y_train_sep, X_test_sep[j])for j in range(X_test_sep.shape[0])])
    
    acc = (y_pred == Y_test_sep).mean()*100
    taux_erreur.append(100 - acc)

plt.plot(range(1, k_max), taux_erreur, color="blue")
plt.xlabel("k")
plt.ylabel("Taux d'erreur (%)")
plt.show()

In [ ]:
#apres on applique l'algorithme sur prev
y_test=np.array([classifie_kppv(17, x_train, y_train, x_test[i]) for i in range(x_test.shape[0])])

In [ ]:
test_ids = prev_data.index

submission = pd.DataFrame({"Id": test_ids, "espece": y_test})

submission.to_csv(OUTPUT_DIR / "kppv3.csv", index=False)